<a href="https://colab.research.google.com/github/ishach20-a11y/Master-thesis/blob/code/Gemini_zero_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# experiment_runner_gemini_zero_shot.py

# ── 0. Setup ───────────────────────────────────────────────
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import files
from google.colab import userdata
import os
import re

os.makedirs("experiments/dmn", exist_ok=True)

# ── 1. API key and model ───────────────────────────────────
API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3-flash-preview"

# ── 2. Configuration ───────────────────────────────────────
# Change manually per run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 1
MAX_TOKENS = 30000
TOP_P = 1.0

# ── 3. New description ─────────────────────────────────────
description_id = "description_1"

description = """The BMI of a person can be calculated based on weight in kgs and length of a person in meters by using the following formula: weight/(length*length). If the BMI value is above 30, then the BMI-level is considered Obese. If you are a male and the BMI-value is under 18.5 then the BMI-level is severely underweight and if you are female, then you are considered underweight with the same bmi-value. If the BMI-value is between 18.5 and 25 (without 25), then the BMI-level is underweight for a male and normal for a female. Lastly, if the BMI-value is between 25 and 30 and you are a Male then the BMI-level is normal but if you are a female then BMI-level is overweight."""

# ── 4. Zero-shot prompt ────────────────────────────────────
def build_zero_shot_prompt(description: str) -> str:
    return f"""<s>[INST] You are an expert in Camunda-compatible DMN XML generation.

Generate a complete DMN XML file for the textual description below.

Strict output rules:
- Return only DMN XML.
- Do not include explanations, headings, labels, Markdown, or comments.
- The XML must start with <?xml version="1.0" encoding="UTF-8"?>
- The XML must end with </definitions>.

Required namespaces:
- Use DMN namespace: https://www.omg.org/spec/DMN/20191111/MODEL/
- Use DMNDI namespace: https://www.omg.org/spec/DMN/20191111/DMNDI/
- Use DI namespace: http://www.omg.org/spec/DMN/20180521/DI/
- Use DC namespace: http://www.omg.org/spec/DMN/20180521/DC/

Camunda DMN syntax rules:
- For inputData, use <variable name="..." typeRef="..."/>. Do not use <variableType>.
- For decision output variables, use <variable name="..." typeRef="..."/>. Do not use <variableType>.
- In decisionTable input clauses, use <inputExpression id="..." typeRef="..."><text>...</text></inputExpression>.
- In rule inputEntry elements, use only <text>...</text>. Do not wrap inputEntry values in <literalExpression>.
- In rule outputEntry elements, use only <text>...</text>. Do not wrap outputEntry values in <literalExpression>.
- Use <literalExpression id="..." typeRef="..."><text>...</text></literalExpression> only for literal-expression decisions, not inside decision table rules.
- Use preferredOrientation="Rule-as-Row".
- Every decision table rule must contain exactly one inputEntry for each input column.
- Use Camunda-compatible FEEL syntax.
- For half-open intervals, use [18.5..25[ instead of [18.5..25).

Structure rules:
- Every decision must have a unique id.
- Every inputData must have a unique id.
- Every informationRequirement must have a unique id.
- Every informationRequirement must reference an existing decision or inputData.
- Every dmndi:DMNEdge dmnElementRef must exactly match an existing informationRequirement id.
- Do not invent edge ids such as _input1, _input2, edge1, edge2.

DMNDI diagram rules:
- Include a complete <dmndi:DMNDI> section.
- Include one dmndi:DMNShape for every decision and inputData element.
- Include one dmndi:DMNEdge for every informationRequirement.
- Every dmndi:DMNShape must reference an existing decision or inputData id.
- Every dc:Bounds inside a dmndi:DMNShape must include x, y, width, and height.
- Do not include <di:Extension>, <di:extension>, <camunda:Bounds>, or standalone <dc:Bounds> directly inside dmndi:DMNDiagram.

Output quality rules:
- Keep the XML compact.
- Do not add unnecessary metadata.
- Do not generate placeholder or incomplete elements.
- Ensure all XML tags are properly closed before returning the answer.

Text: '{description}' [/INST]</s>"""

# ── 5. Clean model output ──────────────────────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()

    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()

# ── 6. Run generation ──────────────────────────────────────
prompt = build_zero_shot_prompt(description)

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Gemini zero-shot | {description_id} | temp={TEMPERATURE} | iter={iteration}")

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
            max_output_tokens=MAX_TOKENS,
            top_p=TOP_P,
        ),
    )

    #Checking the amount of tokens used per runing of the code
    usage = response.usage_metadata

    print("Input tokens:", usage.prompt_token_count)
    print("Output tokens:", usage.candidates_token_count)
    print("Total tokens:", usage.total_token_count)

    dmn_xml = clean_model_output(response.text)

    base_name = f"{description_id}_gemini_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

# ── 7. Download DMN results ────────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    base_name = f"{description_id}_gemini_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    files.download(f"experiments/dmn/{base_name}.dmn")

▶ Gemini zero-shot | description_1 | temp=0.2 | iter=1
Input tokens: 923
Output tokens: 1200
Total tokens: 30919
Saved: experiments/dmn/description_1_gemini_zero_shot_temp_0.2_iter_1.dmn


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>